In [1]:
import sys
from pathlib import Path
import numpy as np, pandas as pd
from sklearn.metrics import roc_auc_score

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))
from src.evaluation import evaluate
from src.pipeline import DATA, RESULTS

SC, WIN = RESULTS / "scores", DATA / "windows"
SPLITS = ["val", "tune", "test"]
flow_files = {
    "M1 IsolationForest": "m1_iforest_nftoniot_seed11.parquet",
    "M2 OneClassSVM": "m2_ocsvm_nftoniot_seed11.parquet",
    "M3 AE v1": "m3_autoencoder_nftoniot_seed11.parquet",
    "M3 AE v2": "m3v2_autoencoder_nftoniot_seed11.parquet",
}

def window_meta(sp):
    z = np.load(WIN / f"{sp}.npz", allow_pickle=True)
    return pd.DataFrame({"id": z["win_ids"], "label": z["label"].astype(int),
                         "attack_type": z["attack_type"].astype(str), "split": sp})

meta = {sp: window_meta(sp) for sp in SPLITS}
flowmap = {sp: pd.read_parquet(WIN / f"{sp}_flowmap.parquet") for sp in SPLITS}

def to_windows(df_flow, how):
    out, cov = {}, {}
    for sp in SPLITS:
        s = df_flow[df_flow["split"] == sp][["id", "score"]]
        j = flowmap[sp].merge(s, left_on="row_id", right_on="id", how="inner")
        agg = j.groupby("win_id")["score"].agg(how)
        m = meta[sp].copy()
        m["score"] = m["id"].map(agg)
        out[sp] = m.dropna(subset=["score"])
        cov[sp] = round(len(j) / len(flowmap[sp]), 3)     # share of window flows that have a score
    return out, cov

models, coverage = {}, {}
for name, f in flow_files.items():
    df = pd.read_parquet(SC / f)
    for how in ["mean", "max"]:
        models[f"{name} ({how})"], coverage[f"{name} ({how})"] = to_windows(df, how)

m4 = pd.read_parquet(SC / "m4_tcn_nftoniot_seed11.parquet")
models["M4 TCN"] = {sp: m4[m4["split"] == sp] for sp in SPLITS}
coverage["M4 TCN"] = {sp: 1.0 for sp in SPLITS}
print("coverage (share of window flows that have a baseline score):")
print(pd.DataFrame(coverage).T)

coverage (share of window flows that have a baseline score):
                           val  tune   test
M1 IsolationForest (mean)  1.0   1.0  0.524
M1 IsolationForest (max)   1.0   1.0  0.524
M2 OneClassSVM (mean)      1.0   1.0  0.524
M2 OneClassSVM (max)       1.0   1.0  0.524
M3 AE v1 (mean)            1.0   1.0  0.524
M3 AE v1 (max)             1.0   1.0  0.524
M3 AE v2 (mean)            1.0   1.0  0.524
M3 AE v2 (max)             1.0   1.0  0.524
M4 TCN                     1.0   1.0  1.000


In [2]:
rows = []
for name, d in models.items():
    for sp in ["tune", "test"]:
        r, o = evaluate(d[sp], val_df=d["val"]), evaluate(d[sp])
        rows.append({"model": name, "split": sp, "auroc": r["auroc"],
                     "macro_tpr@5%": r["macro_tpr@fpr0.05"], "real_fpr@5%": r["real_fpr@0.05"],
                     "oracle_macro_tpr@5%": o["macro_tpr@fpr0.05"]})
cmp_table = pd.DataFrame(rows).round(3)
cmp_table.to_csv(RESULTS / "window_level_comparison.csv", index=False)
cmp_table[cmp_table["split"] == "tune"]

,model,split,auroc,macro_tpr@5%,real_fpr@5%,oracle_macro_tpr@5%
0,M1 IsolationForest (mean),tune,0.885,0.221,0.047,0.223
2,M1 IsolationForest (max),tune,0.654,0.468,0.071,0.356
4,M2 OneClassSVM (mean),tune,0.915,0.407,0.040,0.476
6,M2 OneClassSVM (max),tune,0.794,0.211,0.049,0.223
8,M3 AE v1 (mean),tune,0.910,0.387,0.051,0.377
10,M3 AE v1 (max),tune,0.841,0.271,0.050,0.271
12,M3 AE v2 (mean),tune,0.958,0.583,0.049,0.598
14,M3 AE v2 (max),tune,0.924,0.239,0.040,0.357
16,M4 TCN,tune,0.571,0.580,0.106,0.279


In [3]:
def per_attack_auroc(x):
    b = x[x["label"] == 0]["score"].to_numpy()
    return {t: roc_auc_score(np.r_[np.zeros(len(b)), np.ones(len(g))], np.r_[b, g["score"].to_numpy()])
            for t, g in x[x["label"] == 1].groupby("attack_type")}

show = [k for k in models if "max" not in k]
tbl = {(k, sp): per_attack_auroc(models[k][sp]) for k in show for sp in ["tune", "test"]}
pd.DataFrame(tbl).round(3)

M1 IsolationForest (mean)        M2 OneClassSVM (mean)         \
                                tune   test                  tune   test   
ddos                           0.891  0.623                 0.918  0.529   
dos                            0.879  0.443                 0.950  0.415   
injection                      0.960  0.832                 0.960  0.334   
mitm                           0.844  0.543                 0.917  0.428   
password                       0.926  0.826                 0.910  0.428   
scanning                       0.872  0.200                 0.909  0.140   
xss                            0.910  0.628                 0.953  0.237   
backdoor                         NaN  0.689                   NaN  0.685   
ransomware                       NaN  0.762                   NaN  0.251   

           M3 AE v1 (mean)        M3 AE v2 (mean)        M4 TCN         
                      tune   test            tune   test   tune   test  
ddos                 0.925  0.334           0.928  0.347  0.877  0.478  
dos                  0.938  0.428           0.942  0.320  0.913  0.668  
injection            0.969  0.416           0.972  0.235  0.918  0.324  
mitm                 0.899  0.477           0.906  0.409  0.883  0.587  
password             0.939  0.504           0.948  0.259  0.910  0.451  
scanning             0.897  0.171           0.962  0.154  0.448  0.287  
xss                  0.943  0.261           0.947  0.190  0.932  0.606  
backdoor               NaN  0.860             NaN  0.925    NaN  0.627  
ransomware             NaN  0.720             NaN  0.458    NaN  0.359

In [4]:
def make_fused(a, b, how="max", eps=1e-6):
    A, B = models[a], models[b]
    stats = {}
    for tag, M in [("a", A), ("b", B)]:
        lv = np.log(M["val"].query("label == 0")["score"].to_numpy() + eps)
        stats[tag] = (lv.mean(), lv.std())

    out = {}
    for sp in ["val", "tune", "test"]:
        j = A[sp][["id", "label", "attack_type", "split", "score"]].merge(
            B[sp][["id", "score"]].rename(columns={"score": "sb"}), on="id")
        za = (np.log(j["score"].to_numpy() + eps) - stats["a"][0]) / stats["a"][1]
        zb = (np.log(j["sb"].to_numpy() + eps) - stats["b"][0]) / stats["b"][1]
        j["score"] = np.maximum(za, zb) if how == "max" else (za + zb) / 2
        out[sp] = j[["id", "label", "attack_type", "split", "score"]]
    return out

for how in ["max", "mean"]:
    models[f"M4 + M3v2 ({how})"] = make_fused("M4 TCN", "M3 AE v2 (max)", how)

rows = []
for name in ["M3 AE v2 (max)", "M4 TCN", "M4 + M3v2 (max)", "M4 + M3v2 (mean)"]:
    d = models[name]
    for sp in ["tune", "test"]:
        r, o = evaluate(d[sp], val_df=d["val"]), evaluate(d[sp])
        rows.append({"model": name, "split": sp, "auroc": r["auroc"],
                     "macro_tpr@5%": r["macro_tpr@fpr0.05"], "real_fpr@5%": r["real_fpr@0.05"],
                     "oracle_macro_tpr@5%": o["macro_tpr@fpr0.05"]})
fusion = pd.DataFrame(rows).round(3)
fusion.to_csv(RESULTS / "fusion_comparison.csv", index=False)
print(fusion.to_string(index=False))
print(pd.DataFrame({n: per_attack_auroc(models[n]["tune"])
                    for n in ["M3 AE v2 (max)", "M4 TCN", "M4 + M3v2 (max)"]}).round(3))

           model split  auroc  macro_tpr@5%  real_fpr@5%  oracle_macro_tpr@5%
  M3 AE v2 (max)  tune  0.924         0.239        0.040                0.357
  M3 AE v2 (max)  test  0.228         0.431        0.814                0.036
          M4 TCN  tune  0.571         0.580        0.106                0.279
          M4 TCN  test  0.480         0.687        0.708                0.002
 M4 + M3v2 (max)  tune  0.894         0.388        0.068                0.314
 M4 + M3v2 (max)  test  0.309         0.644        0.821                0.032
M4 + M3v2 (mean)  tune  0.839         0.543        0.074                0.435
M4 + M3v2 (mean)  test  0.347         0.726        0.750                0.003
           M3 AE v2 (max)  M4 TCN  M4 + M3v2 (max)
ddos                0.919   0.877            0.907
dos                 0.939   0.913            0.929
injection           0.967   0.918            0.953
mitm                0.908   0.883            0.876
password            0.935   0.910          